In [ ]:
#Objective: To build and evaluate ML models to predict customer churn using telco customer churn dataset.
#Steps include: preprocessing, encoding, applying feature engineering to improve predictions and avoid redundancy.
#Training two models: random forest classifier and logistic regression
#Evalating model performance using classigication metrics, confusion matrix and ROC-AUC
import pandas as pd
import numpy as np
import joblib
from google.colab import drive
drive.mount('/content/drive')
from google.colab import files
uploaded = files.upload()
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
#Load + clean data
#df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

df = pd.read_csv('Telco-Customer-Churn-rawdata.csv')
df = df.replace(r'^\s*$', np.nan, regex=True)
df = df.drop(['customerID'], axis=1)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df[df['tenure'] != 0]
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())

df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


#Feature engineering
df['AvgCharges'] = df['TotalCharges'] / df['tenure']
df['AvgCharges'] = df['AvgCharges'].fillna(0)

#Split features/target
X = df.drop('Churn', axis=1)
y = df['Churn']



#Define column types
num_cols = ['tenure','MonthlyCharges','TotalCharges' ]
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

#Build pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000,class_weight='balanced'))
])


# Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# train
pipeline.fit(X_train, y_train)
models = {"Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42) }
#evaluate prediction
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
results = {}
for name, model in models.items():
    print(f"\n{'*'*40}")
    print(f"Model: {name}")
    
    # Create pipeline for each model
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Store results
    results[name] = cm
    
    # Print metrics
    print("Confusion Matrix:\n", cm)
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))
    
    # Plot
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, 
                annot=True, 
                fmt='d', 
                cmap='Purples',
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

#key insights
#1. 1. Overall Model Performance
# Logistic Regression performed best in terms of recall (79%), it correctly identified most customers who actually churned. 
# This is important because minimizing missed churn cases (false negatives) is critical in customer retention scenarios.
# Random Forest and Gradient Boosting achieved higher precision (63% and 64%), meaning their churn predictions were more accurate when they did predict churn, 
# but they missed a larger portion of actual churners (higher false negatives).



LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.51      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

Confusion Matrix(logistic regression): 
 [[917 116]
 [182 192]]
ROC-AUC (Logistic): 0.8317397538968064

RANDOM FOREST
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1033
           1       0.66      0.48      0.56       374

    accuracy                           0.80      1407
   macro avg       0.74      0.70      0.71      1407
weighted avg       0.78      0.80      0.78      1407

Confusion Matrix(random forest): 
 [[939  94]
 [194 180]]
ROC-AUC (Random Forest): 0.8313644387615118


['../models/scaler.pkl']